# Docs 1 — The Pipeline, in Miniature

Documents in, answers out — the whole shape in one small run, so you know
where every later lesson fits. Run every cell; change things and rerun.

In [ ]:
# The pile: six monthly reports from a community food pantry.
# Three are clean, three carry the classic damage (headers, spacing, OCR).
pile = {
 "jan.txt": """Date: 2026-01-16
Families served: 167
Donations received: $1,210.00
Contact: pantry@example.org""",
 "feb.txt": """Date: 2026-02-13
Families served: 174
Donations received: $1,385.50
Contact: pantry@example.org""",
 "mar_extracted.txt": """COMMUNITY FOOD PANTRY \u2014 MONTHLY REPORT
Page 1 of 1   PANTRY-MAR-FINAL
Date:  March 14, 2026
Families served:212
Donations received : $1,847.50
Contact: pantry@example.org""",
 "apr_extracted.txt": """Page 1 of 1   PANTRY-APR-FINAL
Date: 4/11/26
Families   served: 198
Donations received: $2,210.00
contact: pantry@example.org""",
 "may_ocr.txt": """Date: May 9, 2026
Families served: 241
Donations received: $l,655.25
Contact: pantry@example.org""",
 "jun_note.txt": """Quick note instead of the form this month, sorry! We had a
great June \u2014 somewhere around fifteen hundred dollars came in between the
two drives, and I counted 188 families across the month. \u2014 Rosa""",
}
for name, text in pile.items():
    print(f"--- {name} ({len(text)} chars)")
    print(text[:120].replace(chr(10), " / "))

## Stage by stage, fast

Each stage below is one honest step. Later lessons rebuild each one
properly — today you watch the shape work.

In [ ]:
import re

def clean(text):
    text = "\n".join(l for l in text.split("\n") if not l.startswith("Page "))
    text = re.sub(r"  +", " ", text)
    text = re.sub(r" ?: ?", ": ", text)
    return text

def structure(text):
    row = {}
    m = re.search(r"[Ff]amilies\s*served:\s*(\d+)", text)
    row["families"] = int(m.group(1)) if m else None
    m = re.search(r"\$([\d,]+\.\d\d)", text)
    row["donations"] = float(m.group(1).replace(",", "")) if m else None
    return row

def validate(row):
    problems = []
    if row["families"] is None or not (0 <= row["families"] <= 5000):
        problems.append("families missing or implausible")
    if row["donations"] is None or row["donations"] <= 0:
        problems.append("donations missing or non-positive")
    return problems

table, quarantine = [], []
for name, raw in pile.items():
    row = structure(clean(raw))
    row["source"] = name
    problems = validate(row)
    (quarantine if problems else table).append((row, problems))

print("TABLE:")
for row, _ in table:
    print("  ", row)
print("QUARANTINE:")
for row, problems in quarantine:
    print("  ", row["source"], "->", problems)

Look at what just happened: four documents made it, two went to
quarantine — the OCR-damaged May (`$l,655.25` is not a number) and Rosa's
free-text June note (no fixed fields for the rules to find). Both are
exactly the cases later lessons handle: validation caught one, and LLM
extraction (lesson 5) will read the other.

## Ask

In [ ]:
total = sum(r["donations"] for r, _ in table)
families = sum(r["families"] for r, _ in table)
print(f"Answer: {len(table)} usable months, {families} families, ${total:,.2f} donated")
print("Rows used:", [r["source"] for r, _ in table])
print("NOT included (quarantined):", [r["source"] for r, _ in quarantine])

The answer names its rows AND its exclusions — that honesty is the
pipeline's product as much as the numbers are.

## Turn-in

Describe a real document pile you could process by lesson 8: what's in it,
what formats, and three questions it could answer as a table.

In [ ]:
my_pile = """
The pile:
Formats:
Question 1:
Question 2:
Question 3:
"""
print(my_pile)